# Introduction to Vector Databases (Installing and Running ChromaDB)

**Level:** Scratch → Intermediate
**Runs on:** Google Colab (Free CPU)

### What you will learn
1. What a Vector Database is and why we need one
2. Installing ChromaDB (free, open-source, runs locally)
3. Creating a Collection (like a table, but for vectors)
4. Adding documents (ChromaDB auto-generates embeddings)
5. Querying by meaning (semantic search)
6. Using your own Hugging Face embeddings with ChromaDB (intermediate)
7. Updating and deleting records

> No API key, no cloud account, no cost. Everything runs in-memory inside this notebook.

## 1. Why a Vector Database?

A normal database (SQL) searches by **exact match** — e.g., `WHERE name = 'dog'`.

A **Vector Database** stores embeddings (numeric meaning-vectors) and searches by **similarity** — e.g., "find text that *means* something like 'dog'", even if the word "dog" never appears.

| Normal Database | Vector Database |
|---|---|
| Matches exact text/values | Matches by meaning |
| Uses SQL queries | Uses similarity search |
| Good for structured data | Good for text, images, audio embeddings |

**ChromaDB** is one of the simplest free, open-source vector databases — great for learning.

### 2. Install ChromaDB
Run this once. `-q` keeps the output clean.

In [12]:
!pip install chromadb sentence-transformers

In [1]:
!pip install -q chromadb

### 3. Create a ChromaDB Client

We use an **in-memory / ephemeral client** — perfect for learning in Colab.
(Data will reset when the runtime restarts. For permanent storage, `chromadb.PersistentClient(path="...")` is used instead — shown later.)

In [2]:
import chromadb

client = chromadb.Client()

print("ChromaDB client created successfully!")

ChromaDB client created successfully!


### 4. Create a Collection

A **Collection** in ChromaDB is like a table — it stores your documents, their embeddings, and metadata.

In [3]:
collection = client.create_collection(name="my_first_collection")

print("Collection created:", collection.name)

Collection created: my_first_collection


### 5. Add Documents

We simply give ChromaDB plain text. By default, it **automatically converts text into embeddings** for us using a built-in free model — no extra setup needed.

In [4]:
collection.add(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers.",
        "Paris is the capital city of France."
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5", "doc6"]
)

print("Documents added! Total documents:", collection.count())

Documents added! Total documents: 6


**Discussion Point:** Every document got an `id` (like a primary key). Behind the scenes, ChromaDB converted each sentence into an embedding vector and stored it — just like we did manually in the earlier embeddings notebook.

### 6. Query by Meaning (Semantic Search)

Now we ask a question in natural language. ChromaDB converts our query into an embedding too, then finds the closest matching documents.

In [5]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 0.9317  |  Deep learning uses neural networks with many layers.
Distance: 0.9516  |  Python is a popular programming language for AI.


**Discussion Point:** Lower `distance` = more similar (closer meaning). Notice the AI/ML related sentences come back first, even though the query never used the exact same words.

In [6]:
# Try another query
results = collection.query(
    query_texts=["What is the capital of France?"],
    n_results=2
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 0.3515  |  Paris is the capital city of France.
Distance: 1.7959  |  Python is a popular programming language for AI.


### 7. Adding Metadata (Intermediate)

Real-world documents usually have extra info (category, author, date). ChromaDB lets us store this alongside each document and filter on it.

In [7]:
collection.add(
    documents=[
        "The new smartphone has an amazing camera.",
        "Stock prices rose sharply after the earnings report."
    ],
    metadatas=[
        {"category": "technology"},
        {"category": "finance"}
    ],
    ids=["doc7", "doc8"]
)

# Query only within the "technology" category
results = collection.query(
    query_texts=["Tell me about new gadgets"],
    n_results=2,
    where={"category": "technology"}
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 1.1460  |  The new smartphone has an amazing camera.


### 8. Using Your Own Hugging Face Embeddings (Intermediate)

By default ChromaDB uses its own built-in embedding model. But we can plug in **any** Hugging Face model, such as `all-MiniLM-L6-v2` from the earlier notebook, for full control.

In [8]:
!pip install -q sentence-transformers

In [9]:
from chromadb.utils import embedding_functions

hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create a new collection that uses our chosen Hugging Face model
custom_collection = client.create_collection(
    name="custom_embedding_collection",
    embedding_function=hf_embedder
)

custom_collection.add(
    documents=[
        "I love dogs",
        "I like puppies",
        "The stock market crashed today"
    ],
    ids=["a1", "a2", "a3"]
)

results = custom_collection.query(
    query_texts=["puppy lovers"],
    n_results=2
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Distance: 0.2513  |  I like puppies
Distance: 0.3943  |  I love dogs


### 9. Update and Delete Records

In [10]:
# Update a document's text
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."]
)
print("Updated doc4")

# Delete a document
collection.delete(ids=["doc2"])
print("Deleted doc2")

print("Remaining documents:", collection.count())

Updated doc4
Deleted doc2
Remaining documents: 7


### 10. Persistent Storage (So Data Survives a Restart)

The examples above used an in-memory client, which resets on restart. For real projects, use `PersistentClient` to save data to disk.

In [11]:
persistent_client = chromadb.PersistentClient(path="/content/chroma_db")

persistent_collection = persistent_client.get_or_create_collection(name="persistent_demo")

persistent_collection.add(
    documents=["This data will be saved to disk."],
    ids=["p1"]
)

print("Saved to disk at /content/chroma_db")
print("Count:", persistent_collection.count())

Saved to disk at /content/chroma_db
Count: 1


### 11. Key Takeaways (Recap for Students)

| Concept | Meaning |
|---|---|
| Vector Database | Stores embeddings, searches by meaning/similarity |
| ChromaDB | Free, open-source, runs locally — no cloud needed |
| Collection | Like a table — holds documents + embeddings + metadata |
| `add()` | Insert documents (embeddings created automatically) |
| `query()` | Search by meaning, returns closest matches |
| `where={}` | Filter results using metadata |
| `PersistentClient` | Saves the database to disk instead of memory |

### 12. Practice Exercises
1. Add 5 of your own sentences to a new collection and query them with a related question.
2. Add a `"category"` metadata field to every document, then filter a query by two different categories.
3. Swap the embedding function to `'all-mpnet-base-v2'` and compare the distance scores to `all-MiniLM-L6-v2`.

In [15]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="exercise1"
)

collection.upsert(
    ids=["1", "2", "3", "4", "5"],
    documents=[
        "Python is used for data science and machine learning.",
        "Java is used for software development.",
        "Artificial intelligence helps computers perform intelligent tasks.",
        "Cricket is a popular sport in India.",
        "Healthy food is important for our body."
    ]
)

query = "I want to learn about artificial intelligence."

results = collection.query(
    query_texts=[query],
    n_results=3
)

print("Query:", query)
print("\nTop 3 Results:")

for document, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print("Document:", document)
    print("Distance:", round(distance, 4))
    print()

Query: I want to learn about artificial intelligence.

Top 3 Results:
Document: Artificial intelligence helps computers perform intelligent tasks.
Distance: 0.6029

Document: Python is used for data science and machine learning.
Distance: 1.1508

Document: Java is used for software development.
Distance: 1.5036



In [16]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="exercise2"
)

collection.upsert(
    ids=["1", "2", "3", "4", "5", "6"],
    documents=[
        "Python is used for programming.",
        "Java is used for application development.",
        "Football is a popular sport.",
        "Cricket is popular in India.",
        "Machine learning is a part of AI.",
        "Basketball is a popular sport."
    ],
    metadatas=[
        {"category": "programming"},
        {"category": "programming"},
        {"category": "sports"},
        {"category": "sports"},
        {"category": "AI"},
        {"category": "sports"}
    ]
)

# Search only programming documents
results = collection.query(
    query_texts=["I want to learn programming"],
    n_results=2,
    where={"category": "programming"}
)

print("Programming Results:")

for document in results["documents"][0]:
    print("-", document)

# Search only sports documents
results = collection.query(
    query_texts=["Tell me about sports"],
    n_results=3,
    where={"category": "sports"}
)

print("\nSports Results:")

for document in results["documents"][0]:
    print("-", document)

Programming Results:
- Python is used for programming.
- Java is used for application development.

Sports Results:
- Basketball is a popular sport.
- Football is a popular sport.
- Cricket is popular in India.


In [17]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="./chroma_db")

documents = [
    "Artificial intelligence helps computers perform intelligent tasks.",
    "Machine learning allows computers to learn from data.",
    "Python is a popular programming language.",
    "Cricket is a popular sport in India.",
    "Healthy food is important for our body."
]

ids = ["1", "2", "3", "4", "5"]

# MiniLM embedding function
minilm = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# MPNet embedding function
mpnet = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2"
)

# Create collections
mini_collection = client.get_or_create_collection(
    name="exercise3_minilm",
    embedding_function=minilm
)

mpnet_collection = client.get_or_create_collection(
    name="exercise3_mpnet",
    embedding_function=mpnet
)

# Add documents
mini_collection.upsert(
    ids=ids,
    documents=documents
)

mpnet_collection.upsert(
    ids=ids,
    documents=documents
)

# Query
query = "I want to learn about artificial intelligence."

mini_results = mini_collection.query(
    query_texts=[query],
    n_results=3
)

mpnet_results = mpnet_collection.query(
    query_texts=[query],
    n_results=3
)

print("===== all-MiniLM-L6-v2 =====")

for document, distance in zip(
    mini_results["documents"][0],
    mini_results["distances"][0]
):
    print("Document:", document)
    print("Distance:", round(distance, 4))
    print()

print("===== all-mpnet-base-v2 =====")

for document, distance in zip(
    mpnet_results["documents"][0],
    mpnet_results["distances"][0]
):
    print("Document:", document)
    print("Distance:", round(distance, 4))
    print()
    

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

===== all-MiniLM-L6-v2 =====
Document: Artificial intelligence helps computers perform intelligent tasks.
Distance: 0.3014

Document: Machine learning allows computers to learn from data.
Distance: 0.5422

Document: Python is a popular programming language.
Distance: 0.6318

===== all-mpnet-base-v2 =====
Document: Artificial intelligence helps computers perform intelligent tasks.
Distance: 0.446

Document: Machine learning allows computers to learn from data.
Distance: 0.5587

Document: Python is a popular programming language.
Distance: 0.739

